# Blind search from near-GHZ Acín canonical states

This notebook searches for Bell inequalities that separate a deliberately biased family of **quantum behaviours** from the corrected tripartite cycle/siglocal polytope.

The three-qubit pure state is sampled in the Acín canonical form

$$
|\psi\rangle=
\lambda_0|000\rangle+
\lambda_1e^{i\phi}|100\rangle+
\lambda_2|101\rangle+
\lambda_3|110\rangle+
\lambda_4|111\rangle,
$$

with $\lambda_i\geq0$, $\sum_i\lambda_i^2=1$, and $0\leq\phi\leq\pi$. The departure weight

$$
r=\lambda_1^2+\lambda_2^2+\lambda_3^2
$$

is sampled in four strata near the generalized-GHZ subspace. For every sampled state, input 0 of every party is the computational-basis measurement and each input-1 measurement is independently uniform on the Bloch sphere.

For every generated behaviour $q$, the notebook solves both LPs used in the project:

1. **Fixed-$\beta$ LP**
   $$
   \min_c c\!\cdot q,
   \qquad c\!\cdot v\ge 1\quad\forall v\in\mathrm{vert}(P_{\rm cycle}),
   \qquad c\ge0.
   $$

2. **Free-$\beta$ LP**
   $$
   \min_{c,\beta} c\!\cdot q-\beta,
   \qquad c\!\cdot v\ge\beta\quad\forall v\in\mathrm{vert}(P_{\rm cycle}),
   \qquad c\!\cdot u-\beta=1,
   $$
   where $u(abc|xyz)=1/8$ is white noise.

No previously known inequality or quantum realization is stored in the notebook. Candidates arise only from the seeded blind samples. Every reconstructed inequality is then audited on all 8,128 cycle vertices, including its exact bound, saturating face, and facet status.


## How to use the notebook

Run all cells in order with the Julia kernel. The default research pilot contains **1,000 samples**, with exactly 250 samples in each configured $r$ stratum. For a quick installation test, set `N_NEAR_GHZ_SAMPLES = 4`; this produces one sample per stratum.

The search is blind and does **not** stop when it finds a familiar or promising inequality. It completes the configured batch. Only after the batch is complete should `candidate_ranking.tsv` be used to select inequalities for high-precision reevaluation, NPA, and dedicated multistart see-saws.

Important conventions:

- inputs and outputs are binary;
- the canonical order of all 64 probability coordinates is stated explicitly below;
- the canonical-state sampling measure is purposefully near-GHZ and is **not** Haar measure on all three-qubit pure states;
- fixing input 0 to the computational basis is a search restriction, not a claim made without loss of generality;
- both facets and nonfacets are retained;
- a sampled violation is reproducible from the saved state and measurements, but final quantum optimization is left to a separate notebook;
- if the sample count, strata, seed, or tolerances are changed, use a new output directory. A checkpoint with a different configuration is rejected.


## 1. Install the Julia dependencies if needed

`LinearAlgebra`, `Random`, `Printf`, `Dates`, `Serialization`, and `SHA` are Julia standard libraries. The only external packages are `JuMP`, `MathOptInterface`, and `GLPK`.

In [4]:
import Pkg

required_packages = ["JuMP", "MathOptInterface", "GLPK"]
missing_packages = filter(
    package -> Base.find_package(package) === nothing,
    required_packages,
)

if !isempty(missing_packages)
    println("Installing missing packages: ", join(missing_packages, ", "))
    Pkg.add(missing_packages)
else
    println("All required external Julia packages are already installed.")
end

All required external Julia packages are already installed.


## 2. Imports, coordinate order, and user configuration

The probability index is

$$
j(a,b,c,x,y,z)=1+c+2b+4a+8z+16y+32x.
$$

Thus rows of an $8\times8$ display matrix are ordered by `xyz = 000,001,...,111`, and columns by `abc = 000,001,...,111`.

The default experiment uses 1,000 near-GHZ Acín states. The four $r$ intervals are sampled equally often, in round-robin order, so every four consecutive sample IDs contain one state from each stratum. Within a stratum, $r$ is uniform; the remaining state parameters are sampled as described in Section 5.


In [6]:
using LinearAlgebra
using Random
using Printf
using Dates
using Serialization
using SHA
using JuMP
using GLPK

# Import MathOptInterface directly. This avoids the JuMP.MathOptInterface
# namespace error encountered with Julia 1.12.
import MathOptInterface as MOI

const TRIPLES = [(u, v, w) for u in 0:1 for v in 0:1 for w in 0:1]

prob_index(a, b, c, x, y, z) =
    1 + c + 2*b + 4*a + 8*z + 16*y + 32*x

state_index(a, b, c) = 1 + c + 2*b + 4*a

@assert [prob_index(a, b, c, 0, 0, 0) for (a, b, c) in TRIPLES] == collect(1:8)
@assert [prob_index(0, 0, 0, x, y, z) for (x, y, z) in TRIPLES] == collect(1:8:57)
@assert [state_index(a, b, c) for (a, b, c) in TRIPLES] == collect(1:8)

# -------------------------- USER CONFIGURATION --------------------------

# The default pilot has 250 samples in each of the four strata below.
# For a quick installation test, set this to 4 (one sample per stratum).
# Any configured value must be a positive multiple of the number of strata.
N_NEAR_GHZ_SAMPLES = 40_000

# r = lambda_1^2 + lambda_2^2 + lambda_3^2 is the probability weight outside
# span{|000>,|111>}. Endpoints have probability zero under continuous sampling;
# the generator explicitly samples strictly inside each interval.
R_STRATA = [
    (label="r_0_to_0.01",    lower=0.00, upper=0.01),
    (label="r_0.01_to_0.05", lower=0.01, upper=0.05),
    (label="r_0.05_to_0.15", lower=0.05, upper=0.15),
    (label="r_0.15_to_0.40", lower=0.15, upper=0.40),
]

# Every sample has the deterministic seed MASTER_SEED + sample_id - 1.
# Consequently, a resumed run regenerates the same target exactly.
MASTER_SEED = 42_000_001

# Keep active result files outside OneDrive. If you change the experimental
# configuration, change this directory too rather than reusing a checkpoint.
SEARCH_OUTPUT_DIRECTORY =
    raw"C:\ResearchRuns\cycle_near_ghz_acin_results_40000"

RESUME_SEARCH = true
STOP_ON_ERROR = true
WRITE_TABLES_EVERY = 1_000

# Reconstruction and validation tolerances. Classical validity and facet
# certification are exact integer/rational calculations after reconstruction.
RATIONAL_RECONSTRUCTION_TOLERANCE = 1e-8
LP_CONSISTENCY_TOLERANCE = 2e-7
QUANTUM_AUDIT_TOLERANCE = 2e-11
NORMALIZED_VIOLATION_TOLERANCE = 1e-9

# ------------------------------------------------------------------------

@assert N_NEAR_GHZ_SAMPLES > 0
@assert !isempty(R_STRATA)
@assert N_NEAR_GHZ_SAMPLES % length(R_STRATA) == 0
@assert WRITE_TABLES_EVERY >= 1
@assert all(stratum -> 0 <= stratum.lower < stratum.upper < 1, R_STRATA)
@assert first(R_STRATA).lower == 0.0
for index in 1:(length(R_STRATA) - 1)
    @assert R_STRATA[index].upper == R_STRATA[index + 1].lower
end


## 3. Generate the corrected cycle/siglocal polytope

Each deterministic strategy receives the two inputs available along one orientation of the communication cycle.

- clockwise: $a=f_A(x,z),\ b=f_B(y,x),\ c=f_C(z,y)$;
- counterclockwise: $a=f_A(x,y),\ b=f_B(y,z),\ c=f_C(z,x)$.

There are 4,096 strategies of each orientation. Their intersection consists of the 64 local deterministic behaviours, so the union has $4096+4096-64=8128$ distinct vertices.

In [8]:
"""Evaluate one of the 16 deterministic binary functions of two bits."""
function response(mask::Int, u::Int, v::Int)
    @assert 0 <= mask <= 15
    @assert u in 0:1 && v in 0:1
    return Int((mask >> (2*u + v)) & 1)
end

"""Construct one deterministic 64-component cycle behaviour."""
function deterministic_vertex(
    fA::Int,
    fB::Int,
    fC::Int,
    orientation::Symbol,
)
    vertex = zeros(Int8, 64)

    for (x, y, z) in TRIPLES
        if orientation === :clockwise
            a = response(fA, x, z)
            b = response(fB, y, x)
            c = response(fC, z, y)
        elseif orientation === :counterclockwise
            a = response(fA, x, y)
            b = response(fB, y, z)
            c = response(fC, z, x)
        else
            error("Unknown orientation: $orientation")
        end

        vertex[prob_index(a, b, c, x, y, z)] = 1
    end

    return vertex
end

"""Generate the 4096 deterministic vertices of one orientation."""
function generate_orientation(orientation::Symbol)
    vertices = Matrix{Int8}(undef, 16^3, 64)
    row = 1

    for fA in 0:15, fB in 0:15, fC in 0:15
        vertices[row, :] .= deterministic_vertex(fA, fB, fC, orientation)
        row += 1
    end

    return vertices
end

row_set(vertices) = Set(Tuple(row) for row in eachrow(vertices))

function validate_deterministic_vertices(vertices)
    @assert size(vertices, 2) == 64
    for vertex in eachrow(vertices)
        @assert all(value -> value == 0 || value == 1, vertex)
        for context in 0:7
            block = (8*context + 1):(8*context + 8)
            @assert sum(vertex[block]) == 1
        end
    end
    return true
end

validate_deterministic_vertices (generic function with 1 method)

In [9]:
vertices_clockwise = generate_orientation(:clockwise)
vertices_counterclockwise = generate_orientation(:counterclockwise)

# The 64 local deterministic behaviours occur in both orientations. Removing
# duplicates does not change the convex hull, but it exposes the true count.
vertices_cycle = unique(
    vcat(vertices_clockwise, vertices_counterclockwise);
    dims=1,
)

clockwise_set = row_set(vertices_clockwise)
counterclockwise_set = row_set(vertices_counterclockwise)

validate_deterministic_vertices(vertices_clockwise)
validate_deterministic_vertices(vertices_counterclockwise)
validate_deterministic_vertices(vertices_cycle)

@assert size(vertices_clockwise) == (4096, 64)
@assert size(vertices_counterclockwise) == (4096, 64)
@assert length(intersect(clockwise_set, counterclockwise_set)) == 64
@assert size(vertices_cycle) == (8128, 64)

const UNIFORM_BEHAVIOUR = fill(1//8, 64)

# Every probability coordinate occurs in exactly 1016 vertices. Consequently,
# the centroid of the 8128 distinct vertices is exactly white noise.
@assert all(sum(vertices_cycle; dims=1) .== 1016)
@assert [
    sum(Int(vertices_cycle[i, j]) for i in axes(vertices_cycle, 1)) //
    size(vertices_cycle, 1)
    for j in axes(vertices_cycle, 2)
] == UNIFORM_BEHAVIOUR

println("Corrected cycle polytope")
println("  clockwise vertices:        ", size(vertices_clockwise, 1))
println("  counterclockwise vertices: ", size(vertices_counterclockwise, 1))
println("  intersection:              ", length(intersect(clockwise_set, counterclockwise_set)))
println("  distinct union:            ", size(vertices_cycle, 1))

Corrected cycle polytope
  clockwise vertices:        4096
  counterclockwise vertices: 4096
  intersection:              64
  distinct union:            8128


## 4. Exact dimension and facet certification

Facet status is decided in the 50-dimensional affine hull of the polytope. The routines below compute ranks modulo several large primes and use a Hadamard bound when needed; they do not infer facet status from a floating-point rank alone.

In [11]:
# Pairwise-distinct primes fitting in Int64. Int128 is used for products.
const RANK_PRIMES = Int64[
    1_000_000_007,
    1_000_000_009,
    998_244_353,
    1_004_535_809,
]

mod_multiply(a::Int64, b::Int64, modulus::Int64) =
    Int64(mod(Int128(a) * Int128(b), Int128(modulus)))

function mod_power(base::Int64, exponent::Int64, modulus::Int64)
    result = Int64(1)
    factor = mod(base, modulus)
    power = exponent
    while power > 0
        if isodd(power)
            result = mod_multiply(result, factor, modulus)
        end
        factor = mod_multiply(factor, factor, modulus)
        power >>= 1
    end
    return result
end

"""Rank of an integer matrix over the prime field Z/pZ."""
function rank_mod_prime(
    matrix::AbstractMatrix{<:Integer},
    modulus::Int64,
)
    n_rows, n_columns = size(matrix)
    work = Matrix{Int64}(undef, n_rows, n_columns)

    @inbounds for index in eachindex(matrix)
        work[index] = mod(Int64(matrix[index]), modulus)
    end

    pivot_row = 1
    @inbounds for column in 1:n_columns
        pivot_row > n_rows && break

        pivot = 0
        for row in pivot_row:n_rows
            if work[row, column] != 0
                pivot = row
                break
            end
        end
        pivot == 0 && continue

        if pivot != pivot_row
            for j in 1:n_columns
                work[pivot_row, j], work[pivot, j] =
                    work[pivot, j], work[pivot_row, j]
            end
        end

        inverse_pivot = mod_power(work[pivot_row, column], modulus - 2, modulus)

        for row in (pivot_row + 1):n_rows
            entry = work[row, column]
            entry == 0 && continue
            factor = mod_multiply(entry, inverse_pivot, modulus)
            for j in column:n_columns
                product = mod_multiply(factor, work[pivot_row, j], modulus)
                work[row, j] = Int64(mod(
                    Int128(work[row, j]) - Int128(product),
                    Int128(modulus),
                ))
            end
        end
        pivot_row += 1
    end

    return pivot_row - 1
end

"""Certify the rational rank of an integer matrix."""
function certified_integer_rank(matrix::AbstractMatrix{<:Integer})
    n_rows, n_columns = size(matrix)
    min(n_rows, n_columns) == 0 && return 0

    ranks = [rank_mod_prime(matrix, prime) for prime in RANK_PRIMES]
    certified_rank = maximum(ranks)
    certified_rank == min(n_rows, n_columns) && return certified_rank

    maximum_squared_row_norm = maximum(
        sum(BigInt(matrix[row, column])^2 for column in 1:n_columns)
        for row in 1:n_rows
    )
    maximum_squared_row_norm == 0 && return 0

    next_minor_size = certified_rank + 1
    modulus_product = prod(BigInt(prime) for prime in RANK_PRIMES)
    hadamard_squared_bound = maximum_squared_row_norm^next_minor_size

    modulus_product^2 > hadamard_squared_bound || error(
        "The modular primes are insufficient to certify this rank exactly.",
    )
    return certified_rank
end

function difference_matrix(points::AbstractMatrix{<:Integer})
    n_points, dimension = size(points)
    n_points <= 1 && return zeros(Int64, 0, dimension)
    differences = Matrix{Int64}(undef, n_points - 1, dimension)
    @inbounds for row in 2:n_points, column in 1:dimension
        differences[row - 1, column] =
            Int64(points[row, column]) - Int64(points[1, column])
    end
    return differences
end

function certified_affine_dimension(points::AbstractMatrix{<:Integer})
    n_points = size(points, 1)
    n_points == 0 && return -1
    n_points == 1 && return 0
    return certified_integer_rank(difference_matrix(points))
end

const SIGLOCAL_AFFINE_DIMENSION = certified_affine_dimension(vertices_cycle)
@assert SIGLOCAL_AFFINE_DIMENSION == 50

println("Exact affine dimension: ", SIGLOCAL_AFFINE_DIMENSION)

Exact affine dimension: 50


## 5. Near-GHZ Acín quantum realizations

The Acín coefficients are sampled through

$$
\begin{aligned}
\lambda_0^2 &= (1-r)s, & \lambda_4^2 &= (1-r)(1-s),\\
(\lambda_1^2,\lambda_2^2,\lambda_3^2) &= r(t_1,t_2,t_3),
\end{aligned}
$$

where

- $r$ is uniform inside its assigned stratum;
- $s\sim\mathrm{Uniform}(0,1)$;
- $(t_1,t_2,t_3)\sim\mathrm{Dirichlet}(1,1,1)$;
- $\phi\sim\mathrm{Uniform}(0,\pi)$.

The Dirichlet sample is generated from three independent exponential variables, so no additional Julia package is required. This construction enforces $\sum_i\lambda_i^2=1$ analytically. It is a controlled search distribution concentrated near the generalized-GHZ subspace, not a purported uniform measure on Acín parameter space.

For measurements, input 0 is fixed to the computational basis at all three parties. Each input-1 binary rank-one projective measurement is sampled independently and uniformly on the Bloch sphere. No equality between Alice's, Bob's, and Charlie's input-1 measurements is imposed.


In [13]:
const I2 = Matrix{ComplexF64}(I, 2, 2)
const KET_ZERO = ComplexF64[1, 0]
const KET_ONE = ComplexF64[0, 1]
const COMPUTATIONAL_MEASUREMENT = [
    KET_ZERO * KET_ZERO',
    KET_ONE * KET_ONE',
]

"""Haar-random unit vector in C^dimension."""
function haar_vector(rng::AbstractRNG, dimension::Int)
    vector = randn(rng, dimension) .+ im .* randn(rng, dimension)
    return ComplexF64.(vector ./ norm(vector))
end

"""Haar-random binary rank-one projective qubit measurement."""
function random_projective_measurement(rng::AbstractRNG)
    ket = haar_vector(rng, 2)
    projector_zero = ket * ket'
    projector_one = I2 - projector_zero
    return [projector_zero, projector_one]
end

"""A reproducible draw strictly inside the open unit interval."""
function open_unit_uniform(rng::AbstractRNG)
    value = rand(rng)
    return value == 0.0 ? nextfloat(0.0) : value
end

"""Uniform Dirichlet(1,1,1), generated without Distributions.jl."""
function uniform_simplex_three(rng::AbstractRNG)
    weights = randexp(rng, 3)
    return weights ./ sum(weights)
end

"""Sample r strictly inside the supplied stratum."""
function stratified_r(rng::AbstractRNG, stratum)
    unit_draw = open_unit_uniform(rng)
    value = stratum.lower + (stratum.upper - stratum.lower) * unit_draw
    # Guard against the extremely unlikely event that floating-point rounding
    # maps an interior draw onto one of the nominal interval endpoints.
    value = clamp(value, nextfloat(stratum.lower), prevfloat(stratum.upper))
    @assert stratum.lower < value < stratum.upper
    return value
end

"""Construct the normalized Acín canonical state in the fixed basis order."""
function acin_state(lambda0, lambda1, lambda2, lambda3, lambda4, phi)
    lambdas = Float64[lambda0, lambda1, lambda2, lambda3, lambda4]
    all(value -> value >= 0, lambdas) || error("Acín coefficients must be nonnegative")
    isapprox(sum(abs2, lambdas), 1.0; atol=2e-14, rtol=0.0) ||
        error("Acín coefficients are not normalized")
    0 <= phi <= pi || error("The canonical phase must lie in [0, pi]")

    state = zeros(ComplexF64, 8)
    state[state_index(0, 0, 0)] = lambda0
    state[state_index(1, 0, 0)] = lambda1 * cis(phi)
    state[state_index(1, 0, 1)] = lambda2
    state[state_index(1, 1, 0)] = lambda3
    state[state_index(1, 1, 1)] = lambda4
    return state
end

"""
Generate one near-GHZ Acín realization.

Measurements are stored as measurements[party][input+1][output+1]. The
returned parameter dictionary is saved with the sample and written to TSV.
"""
function near_ghz_acin_realization(rng::AbstractRNG, stratum)
    r = stratified_r(rng, stratum)
    s = open_unit_uniform(rng)
    simplex = uniform_simplex_three(rng)
    phi = pi * rand(rng)

    lambda0 = sqrt((1 - r) * s)
    lambda4 = sqrt((1 - r) * (1 - s))
    lambda1 = sqrt(r * simplex[1])
    lambda2 = sqrt(r * simplex[2])
    lambda3 = sqrt(r * simplex[3])

    state = acin_state(
        lambda0,
        lambda1,
        lambda2,
        lambda3,
        lambda4,
        phi,
    )
    measurements = [
        [deepcopy(COMPUTATIONAL_MEASUREMENT), random_projective_measurement(rng)]
        for _ in 1:3
    ]

    # For this canonical form, the three-tangle is exactly 4*lambda0^2*lambda4^2.
    three_tangle = 4 * lambda0^2 * lambda4^2
    parameters = Dict{String, Any}(
        "r" => r,
        "s" => s,
        "t1" => simplex[1],
        "t2" => simplex[2],
        "t3" => simplex[3],
        "lambda0" => lambda0,
        "lambda1" => lambda1,
        "lambda2" => lambda2,
        "lambda3" => lambda3,
        "lambda4" => lambda4,
        "phi" => phi,
        "three_tangle" => three_tangle,
    )
    return state, measurements, parameters
end

"""Compute p(abc|xyz) from the Born rule in the canonical order."""
function quantum_behavior(state, measurements)
    @assert length(state) == 8
    probabilities = zeros(Float64, 64)
    maximum_imaginary_part = 0.0

    for (x, y, z) in TRIPLES, (a, b, c) in TRIPLES
        operator = kron(
            kron(measurements[1][x + 1][a + 1], measurements[2][y + 1][b + 1]),
            measurements[3][z + 1][c + 1],
        )
        expectation = dot(state, operator * state)
        maximum_imaginary_part = max(
            maximum_imaginary_part,
            abs(imag(expectation)),
        )
        probability = real(expectation)
        probabilities[prob_index(a, b, c, x, y, z)] =
            abs(probability) <= 1e-14 ? 0.0 : probability
    end

    return probabilities, maximum_imaginary_part
end

"""Deterministic structural tests for the state generator."""
function run_acin_generator_smoke_tests()
    balanced_ghz = acin_state(inv(sqrt(2)), 0, 0, 0, inv(sqrt(2)), 0)
    @assert isapprox(sum(abs2, balanced_ghz), 1.0; atol=2e-14, rtol=0.0)
    @assert count(value -> !iszero(value), balanced_ghz) == 2

    for (index, stratum) in enumerate(R_STRATA)
        state, measurements, parameters = near_ghz_acin_realization(
            MersenneTwister(91_000_000 + index),
            stratum,
        )
        @assert isapprox(sum(abs2, state), 1.0; atol=2e-14, rtol=0.0)
        @assert stratum.lower < parameters["r"] < stratum.upper
        @assert isapprox(
            parameters["t1"] + parameters["t2"] + parameters["t3"],
            1.0;
            atol=2e-14,
            rtol=0.0,
        )
        @assert isapprox(
            parameters["lambda1"]^2 +
            parameters["lambda2"]^2 +
            parameters["lambda3"]^2,
            parameters["r"];
            atol=2e-14,
            rtol=0.0,
        )
        for party in 1:3, output in 1:2
            @assert measurements[party][1][output] == COMPUTATIONAL_MEASUREMENT[output]
        end
    end

    state_a, _, parameters_a = near_ghz_acin_realization(
        MersenneTwister(91_999_999),
        R_STRATA[2],
    )
    state_b, _, parameters_b = near_ghz_acin_realization(
        MersenneTwister(91_999_999),
        R_STRATA[2],
    )
    @assert state_a == state_b
    @assert parameters_a == parameters_b
    return true
end

run_acin_generator_smoke_tests()
println("Near-GHZ Acín generator smoke tests passed.")


Near-GHZ Acín generator smoke tests passed.


## 6. Audit every generated quantum behaviour

Before either LP is solved, the notebook checks the state norm, projector properties, measurement completeness, Born probabilities, normalization, and all two-party nonsignalling marginals. A failed audit stops the default run and writes a failure record.

In [15]:
function nonsignalling_residuals(probabilities)
    normalization = 0.0
    ab = 0.0
    ac = 0.0
    bc = 0.0

    for (x, y, z) in TRIPLES
        block_sum = sum(
            probabilities[prob_index(a, b, c, x, y, z)]
            for (a, b, c) in TRIPLES
        )
        normalization = max(normalization, abs(block_sum - 1))
    end

    for x in 0:1, y in 0:1, a in 0:1, b in 0:1
        lhs = sum(probabilities[prob_index(a, b, c, x, y, 0)] for c in 0:1)
        rhs = sum(probabilities[prob_index(a, b, c, x, y, 1)] for c in 0:1)
        ab = max(ab, abs(lhs - rhs))
    end

    for x in 0:1, z in 0:1, a in 0:1, c in 0:1
        lhs = sum(probabilities[prob_index(a, b, c, x, 0, z)] for b in 0:1)
        rhs = sum(probabilities[prob_index(a, b, c, x, 1, z)] for b in 0:1)
        ac = max(ac, abs(lhs - rhs))
    end

    for y in 0:1, z in 0:1, b in 0:1, c in 0:1
        lhs = sum(probabilities[prob_index(a, b, c, 0, y, z)] for a in 0:1)
        rhs = sum(probabilities[prob_index(a, b, c, 1, y, z)] for a in 0:1)
        bc = max(bc, abs(lhs - rhs))
    end

    return (normalization=normalization, ab=ab, ac=ac, bc=bc)
end

"""Return diagnostics and throw if a realization is not a valid quantum model."""
function audit_quantum_realization(
    state,
    measurements,
    probabilities,
    maximum_imaginary_part;
    tolerance=QUANTUM_AUDIT_TOLERANCE,
)
    state_norm_error = abs(real(dot(state, state)) - 1)
    maximum_hermiticity_error = 0.0
    maximum_idempotence_error = 0.0
    maximum_completeness_error = 0.0
    minimum_projector_eigenvalue = Inf

    for party in 1:3, input in 1:2
        projectors = measurements[party][input]
        @assert length(projectors) == 2
        maximum_completeness_error = max(
            maximum_completeness_error,
            opnorm(projectors[1] + projectors[2] - I2),
        )

        for projector in projectors
            maximum_hermiticity_error = max(
                maximum_hermiticity_error,
                opnorm(projector - projector'),
            )
            maximum_idempotence_error = max(
                maximum_idempotence_error,
                opnorm(projector * projector - projector),
            )
            minimum_projector_eigenvalue = min(
                minimum_projector_eigenvalue,
                minimum(eigvals(Hermitian(projector))),
            )
        end
    end

    ns = nonsignalling_residuals(probabilities)
    minimum_probability = minimum(probabilities)
    maximum_probability = maximum(probabilities)

    state_norm_error <= tolerance || error("State normalization audit failed")
    maximum_hermiticity_error <= tolerance || error("Hermiticity audit failed")
    maximum_idempotence_error <= tolerance || error("Projectivity audit failed")
    maximum_completeness_error <= tolerance || error("Measurement completeness audit failed")
    minimum_projector_eigenvalue >= -tolerance || error("A projector is not positive")
    maximum_imaginary_part <= tolerance || error("Born probability has a non-negligible imaginary part")
    minimum_probability >= -tolerance || error("A Born probability is negative")
    maximum_probability <= 1 + tolerance || error("A Born probability exceeds one")
    ns.normalization <= tolerance || error("Probability normalization audit failed")
    maximum((ns.ab, ns.ac, ns.bc)) <= tolerance || error("Nonsignalling audit failed")

    return Dict{String, Any}(
        "state_norm_error" => state_norm_error,
        "measurement_hermiticity_error" => maximum_hermiticity_error,
        "measurement_idempotence_error" => maximum_idempotence_error,
        "measurement_completeness_error" => maximum_completeness_error,
        "minimum_projector_eigenvalue" => minimum_projector_eigenvalue,
        "born_imaginary_error" => maximum_imaginary_part,
        "minimum_probability" => minimum_probability,
        "maximum_probability" => maximum_probability,
        "normalization_residual" => ns.normalization,
        "ab_nonsignalling_residual" => ns.ab,
        "ac_nonsignalling_residual" => ns.ac,
        "bc_nonsignalling_residual" => ns.bc,
    )
end

# End-to-end smoke test: state + projectors -> Born probabilities -> full audit.
let
    smoke_state, smoke_measurements, _ = near_ghz_acin_realization(
        MersenneTwister(92_000_001),
        R_STRATA[1],
    )
    smoke_behavior, smoke_imaginary_error = quantum_behavior(
        smoke_state,
        smoke_measurements,
    )
    audit_quantum_realization(
        smoke_state,
        smoke_measurements,
        smoke_behavior,
        smoke_imaginary_error,
    )
end
println("End-to-end quantum-realization audit passed.")


End-to-end quantum-realization audit passed.


## 7. Build the two reusable LPs

The models contain all 8,128 corrected vertices. Only their objectives change from one quantum target to the next.

The 64-coordinate representation is retained directly. The free-$\beta$ model therefore has the usual affine-gauge freedom in its coefficient vector. Later deduplication uses the exact slack on every cycle vertex, so coefficient vectors differing only by affine equalities are recognized as the same geometric inequality.

In [17]:
function build_fixed_beta_model(vertices)
    number_of_vertices, dimension = size(vertices)
    model = Model(GLPK.Optimizer)
    set_silent(model)

    @variable(model, coefficients[1:dimension] >= 0)
    @constraint(
        model,
        valid_on_cycle[i=1:number_of_vertices],
        sum(vertices[i, j] * coefficients[j] for j in 1:dimension) >= 1,
    )
    @objective(model, Min, sum(coefficients))
    return model, coefficients
end

function build_free_beta_model(vertices, uniform_behavior)
    number_of_vertices, dimension = size(vertices)
    model = Model(GLPK.Optimizer)
    set_silent(model)

    # Variables are unrestricted unless a bound is supplied.
    @variable(model, coefficients[1:dimension])
    @variable(model, beta)
    @constraint(
        model,
        valid_on_cycle[i=1:number_of_vertices],
        sum(vertices[i, j] * coefficients[j] for j in 1:dimension) >= beta,
    )
    @constraint(
        model,
        white_noise_normalization,
        sum(Float64(uniform_behavior[j]) * coefficients[j] for j in 1:dimension) - beta == 1,
    )
    @objective(model, Min, sum(coefficients) - beta)
    return model, coefficients, beta
end

function solve_fixed_beta_target!(model, coefficients, target)
    set_objective_coefficient(model, coefficients, Float64.(target))
    optimize!(model)
    status = termination_status(model)
    status == MOI.OPTIMAL || error("Fixed-beta LP ended with status $status")
    return (
        coefficients=value.(coefficients),
        beta=1.0,
        solver_objective=objective_value(model),
        status=string(status),
    )
end

function solve_free_beta_target!(model, coefficients, beta, target)
    set_objective_coefficient(model, coefficients, Float64.(target))
    set_objective_coefficient(model, beta, -1.0)
    optimize!(model)
    status = termination_status(model)
    status == MOI.OPTIMAL || error("Free-beta LP ended with status $status")
    return (
        coefficients=value.(coefficients),
        beta=value(beta),
        solver_objective=objective_value(model),
        status=string(status),
    )
end

fixed_beta_model, fixed_beta_coefficients =
    build_fixed_beta_model(vertices_cycle)

free_beta_model, free_beta_coefficients, free_beta_beta =
    build_free_beta_model(vertices_cycle, UNIFORM_BEHAVIOUR)

println("Both reusable LP models were built successfully.")

Both reusable LP models were built successfully.


## 8. Exact reconstruction, validity, facet status, and robustness

GLPK returns floating-point coefficient vectors. For every solution the notebook:

1. rationally reconstructs all coefficients (and $\beta$ in the free LP);
2. clears denominators;
3. computes the **exact** minimum over all 8,128 vertices and uses it as the certified bound $B$;
4. requires that the reconstructed target objective agrees with the LP objective;
5. reduces $(C,B)$ to primitive integers;
6. evaluates the quantum target and white noise;
7. certifies the dimension of the saturating face.

For $C\cdot p\ge B$, define

$$
\Delta=B-C\cdot q,\qquad
W=C\cdot u-B,\qquad
G=\frac{\Delta}{W}.
$$

When $\Delta>0$, the white-noise fraction tolerated by this particular target is

$$
\eta=\frac{G}{1+G}.
$$

These quantities are invariant under positive rescaling and under affine-gauge changes on the polytope.

In [19]:
function rationalize_scalar(value; tolerance=RATIONAL_RECONSTRUCTION_TOLERANCE)
    isfinite(value) || error("Cannot rationalize a non-finite LP coefficient")
    return abs(value) <= tolerance ?
        BigInt(0)//BigInt(1) :
        rationalize(BigInt, value; tol=tolerance)
end

function rationalize_solution(coefficients_float, beta_float)
    coefficients = Rational{BigInt}[
        rationalize_scalar(value) for value in coefficients_float
    ]
    beta = rationalize_scalar(beta_float)
    return coefficients, beta
end

"""Clear denominators simultaneously from coefficients and beta."""
function integer_form(coefficients, beta)
    all_values = vcat(coefficients, [beta])
    common_denominator = foldl(
        lcm,
        denominator.(all_values);
        init=BigInt(1),
    )
    integers = BigInt[
        numerator(value * common_denominator) for value in all_values
    ]
    return integers[1:end-1], integers[end], common_denominator
end

"""Evaluate an integer coefficient vector on every deterministic vertex."""
function integer_vertex_values(vertices, coefficients)
    number_of_vertices, dimension = size(vertices)
    @assert length(coefficients) == dimension
    values = Vector{BigInt}(undef, number_of_vertices)

    @inbounds for row in 1:number_of_vertices
        total = BigInt(0)
        for column in 1:dimension
            if vertices[row, column] != 0
                total += coefficients[column]
            end
        end
        values[row] = total
    end
    return values
end

function primitive_integer_inequality(coefficients, bound)
    common_divisor = foldl(
        gcd,
        abs.(vcat(coefficients, [bound]));
        init=BigInt(0),
    )
    common_divisor == 0 && error("The reconstructed inequality is zero")
    return coefficients .÷ common_divisor, bound ÷ common_divisor
end

function relative_agreement(actual, expected; tolerance=LP_CONSISTENCY_TOLERANCE)
    return abs(actual - expected) <= tolerance * (1 + abs(expected))
end

"""
Return an exact, gauge- and scale-invariant key from all vertex slacks.
The full string is used as the dictionary key; SHA-256 is only a readable ID.
"""
function canonical_slack_key(vertex_values, bound)
    slacks = vertex_values .- bound
    minimum(slacks) == 0 || error("A certified bound must be saturated")
    minimum(slacks) >= 0 || error("The reconstructed inequality is invalid")

    nonzero_slacks = filter(value -> !iszero(value), slacks)
    isempty(nonzero_slacks) && return nothing, nothing
    divisor = foldl(gcd, abs.(nonzero_slacks); init=BigInt(0))
    canonical_slacks = slacks .÷ divisor
    key = join(string.(canonical_slacks), ",")
    identifier = bytes2hex(sha256(codeunits(key)))
    return key, identifier
end

"""Exact affine dimension of a proper face; facet detection can stop at 49."""
function certified_face_dimension(points)
    size(points, 1) == 0 && return -1
    size(points, 1) == 1 && return 0
    differences = difference_matrix(points)

    # Because the inequality is valid, tight, and nonconstant on a
    # 50-dimensional polytope, this face has dimension at most 49. Witnessing
    # rank 49 modulo one prime therefore certifies a facet immediately.
    first_rank = rank_mod_prime(differences, first(RANK_PRIMES))
    first_rank == SIGLOCAL_AFFINE_DIMENSION - 1 && return first_rank

    return certified_integer_rank(differences)
end

certified_face_dimension

In [20]:
"""Audit one numerical LP solution and return a self-contained outcome."""
function audit_lp_solution(
    lp_kind::String,
    numerical_solution,
    target,
    vertices_cycle,
    vertices_clockwise,
    vertices_counterclockwise,
)
    coefficients_rational, beta_rational = rationalize_solution(
        numerical_solution.coefficients,
        numerical_solution.beta,
    )
    coefficients_raw, beta_raw, common_denominator = integer_form(
        coefficients_rational,
        beta_rational,
    )

    if lp_kind == "fixed_beta"
        minimum(coefficients_raw) >= 0 || error(
            "Fixed-beta reconstruction produced a negative coefficient",
        )
    elseif lp_kind != "free_beta"
        error("Unknown LP kind: $lp_kind")
    end

    cycle_values_raw = integer_vertex_values(vertices_cycle, coefficients_raw)
    exact_bound_raw = minimum(cycle_values_raw)
    reconstructed_bound_float = Float64(exact_bound_raw // common_denominator)

    # Check that rational reconstruction preserves the solver's active bound.
    expected_beta = numerical_solution.beta
    relative_agreement(reconstructed_bound_float, expected_beta) || error(
        "Reconstructed classical bound disagrees with the LP beta",
    )

    reconstructed_target_linear = sum(
        Float64(coefficients_raw[j] // common_denominator) * target[j]
        for j in eachindex(target)
    )
    reconstructed_solver_objective = if lp_kind == "fixed_beta"
        reconstructed_target_linear
    else
        reconstructed_target_linear - Float64(beta_raw // common_denominator)
    end
    relative_agreement(
        reconstructed_solver_objective,
        numerical_solution.solver_objective,
    ) || error("Rational reconstruction changed the LP objective")

    coefficients, bound = primitive_integer_inequality(
        coefficients_raw,
        exact_bound_raw,
    )

    cycle_values = integer_vertex_values(vertices_cycle, coefficients)
    clockwise_values = integer_vertex_values(vertices_clockwise, coefficients)
    counterclockwise_values = integer_vertex_values(vertices_counterclockwise, coefficients)
    exact_bound = minimum(cycle_values)
    exact_bound == bound || error("Primitive reduction changed the bound")
    minimum(cycle_values .- bound) == 0 || error("The exact inequality is not tight")
    minimum(cycle_values .- bound) >= 0 || error("The exact inequality is invalid")

    uniform_slack = sum(coefficients) // 8 - bound
    target_value = sum(Float64(coefficients[j]) * target[j] for j in eachindex(target))
    gap = Float64(bound) - target_value

    # In the fixed-beta LP an interior target can return an affine equality,
    # such as normalization. It is valid but is not a Bell inequality and has
    # no white-noise normalization. Record it as a noncandidate LP outcome.
    if uniform_slack == 0
        all(cycle_values .== bound) || error(
            "Zero uniform slack is inconsistent with the vertex centroid",
        )
        return Dict{String, Any}(
            "lp_kind" => lp_kind,
            "status" => numerical_solution.status,
            "solver_objective" => numerical_solution.solver_objective,
            "outcome_kind" => "affine_equality",
            "candidate" => nothing,
            "target_value" => target_value,
            "gap" => gap,
            "normalized_gap" => NaN,
            "critical_visibility" => NaN,
            "critical_noise_fraction" => NaN,
            "target_violates" => false,
        )
    end

    uniform_slack > 0 || error("The reconstructed inequality has negative white-noise slack")
    normalized_gap = gap / Float64(uniform_slack)
    target_violates = normalized_gap > NORMALIZED_VIOLATION_TOLERANCE
    critical_visibility = target_violates ? 1 / (1 + normalized_gap) : NaN
    critical_noise_fraction = target_violates ? normalized_gap / (1 + normalized_gap) : NaN

    saturating_indices = findall(==(bound), cycle_values)
    isempty(saturating_indices) && error("The reconstructed inequality is not saturated")
    face_dimension = certified_face_dimension(vertices_cycle[saturating_indices, :])

    key, candidate_id = canonical_slack_key(cycle_values, bound)
    key === nothing && error("A nonconstant inequality has a constant slack vector")

    candidate = Dict{String, Any}(
        "candidate_id" => candidate_id,
        "canonical_slack_key" => key,
        "coefficients" => coefficients,
        "bound" => bound,
        "uniform_slack" => uniform_slack,
        "cycle_minimum" => minimum(cycle_values),
        "clockwise_minimum" => minimum(clockwise_values),
        "counterclockwise_minimum" => minimum(counterclockwise_values),
        "saturation_count" => length(saturating_indices),
        "face_dimension" => face_dimension,
        "is_facet" => face_dimension == SIGLOCAL_AFFINE_DIMENSION - 1,
    )

    return Dict{String, Any}(
        "lp_kind" => lp_kind,
        "status" => numerical_solution.status,
        "solver_objective" => numerical_solution.solver_objective,
        "outcome_kind" => "inequality",
        "candidate" => candidate,
        "target_value" => target_value,
        "gap" => gap,
        "normalized_gap" => normalized_gap,
        "critical_visibility" => critical_visibility,
        "critical_noise_fraction" => critical_noise_fraction,
        "target_violates" => target_violates,
    )
end

audit_lp_solution

## 9. Checkpoints and human-readable output files

The binary checkpoint is written atomically after every completed sample. The TSV files are rewritten atomically every few samples and at the end. They include:

- every state parameter ($r,s,t_i,\lambda_i,\phi$, and the three-tangle), stratum assignment, seed, and audit residual;
- every LP result, including nonviolations and affine equalities;
- every distinct exact inequality, with facets and nonfacets retained;
- a ranking by the best target-specific white-noise tolerance found in this blind batch;
- every state amplitude, projector matrix, and probability needed to reproduce each sampled quantum behaviour;
- failures, if any.

Deduplication is exact on the complete normalized slack vector over the 8,128 vertices. It does not use rounded floating-point coefficients and does not identify symmetry-related but genuinely different slack vectors.


In [22]:
function rational_string(value::Rational)
    denominator(value) == 1 && return string(numerator(value))
    return "$(numerator(value))/$(denominator(value))"
end

float_string(value) = isfinite(Float64(value)) ? @sprintf("%.17g", Float64(value)) : ""
integer_vector_string(values) = join(string.(values), ",")
safe_text(value) = replace(string(value), '\t' => ' ', '\n' => ' ')

function atomic_write(writer, path)
    temporary_path = path * ".tmp"
    open(temporary_path, "w") do io
        writer(io)
    end
    mv(temporary_path, path; force=true)
    return path
end

function write_checkpoint(path, state)
    return atomic_write(path) do io
        serialize(io, state)
    end
end

function read_checkpoint(path)
    return open(path, "r") do io
        deserialize(io)
    end
end

function append_failure(path, sample_id, ensemble, stage, error_value)
    new_file = !isfile(path)
    open(path, "a") do io
        if new_file
            println(io, "timestamp\tsample_id\tensemble\tstage\tmessage")
        end
        println(io, join([
            string(now()),
            string(sample_id),
            string(ensemble),
            string(stage),
            safe_text(sprint(showerror, error_value)),
        ], '\t'))
    end
end

function output_paths(output_directory)
    return Dict(
        "checkpoint" => joinpath(output_directory, "search_checkpoint.jls"),
        "samples" => joinpath(output_directory, "samples.tsv"),
        "lp_runs" => joinpath(output_directory, "lp_runs.tsv"),
        "candidates" => joinpath(output_directory, "unique_candidates.tsv"),
        "ranking" => joinpath(output_directory, "candidate_ranking.tsv"),
        "states" => joinpath(output_directory, "quantum_states.tsv"),
        "measurements" => joinpath(output_directory, "quantum_measurements.tsv"),
        "behaviors" => joinpath(output_directory, "quantum_behaviors.tsv"),
        "failures" => joinpath(output_directory, "failures.tsv"),
        "metadata" => joinpath(output_directory, "run_metadata.txt"),
    )
end

output_paths (generic function with 1 method)

In [23]:
function write_sample_table(path, samples)
    headers = [
        "sample_id", "ensemble", "ensemble_index", "seed",
        "r_stratum_index", "r_stratum_label", "stratum_draw_index",
        "r_lower", "r_upper", "r", "s", "t1", "t2", "t3",
        "lambda0", "lambda1", "lambda2", "lambda3", "lambda4",
        "phi", "three_tangle",
        "state_norm_error", "measurement_hermiticity_error",
        "measurement_idempotence_error", "measurement_completeness_error",
        "minimum_projector_eigenvalue", "born_imaginary_error",
        "minimum_probability", "maximum_probability", "normalization_residual",
        "ab_nonsignalling_residual", "ac_nonsignalling_residual",
        "bc_nonsignalling_residual",
    ]
    atomic_write(path) do io
        println(io, join(headers, '\t'))
        for sample in samples
            audit = sample["audit"]
            parameters = sample["parameters"]
            println(io, join([
                string(sample["sample_id"]), string(sample["ensemble"]),
                string(sample["ensemble_index"]), string(sample["seed"]),
                string(sample["r_stratum_index"]), string(sample["r_stratum_label"]),
                string(sample["stratum_draw_index"]),
                float_string(sample["r_lower"]), float_string(sample["r_upper"]),
                float_string(parameters["r"]), float_string(parameters["s"]),
                float_string(parameters["t1"]), float_string(parameters["t2"]),
                float_string(parameters["t3"]),
                float_string(parameters["lambda0"]), float_string(parameters["lambda1"]),
                float_string(parameters["lambda2"]), float_string(parameters["lambda3"]),
                float_string(parameters["lambda4"]), float_string(parameters["phi"]),
                float_string(parameters["three_tangle"]),
                float_string(audit["state_norm_error"]),
                float_string(audit["measurement_hermiticity_error"]),
                float_string(audit["measurement_idempotence_error"]),
                float_string(audit["measurement_completeness_error"]),
                float_string(audit["minimum_projector_eigenvalue"]),
                float_string(audit["born_imaginary_error"]),
                float_string(audit["minimum_probability"]),
                float_string(audit["maximum_probability"]),
                float_string(audit["normalization_residual"]),
                float_string(audit["ab_nonsignalling_residual"]),
                float_string(audit["ac_nonsignalling_residual"]),
                float_string(audit["bc_nonsignalling_residual"]),
            ], '\t'))
        end
    end
end

function write_lp_run_table(path, runs)
    headers = [
        "sample_id", "ensemble", "seed", "lp_kind", "solver_status",
        "solver_objective", "outcome_kind", "candidate_id", "is_duplicate",
        "classical_bound", "target_value", "gap", "uniform_slack",
        "normalized_gap", "target_violates", "critical_visibility",
        "critical_noise_fraction", "saturation_count", "face_dimension", "is_facet",
    ]
    atomic_write(path) do io
        println(io, join(headers, '\t'))
        for run in runs
            println(io, join([
                string(run["sample_id"]), string(run["ensemble"]), string(run["seed"]),
                run["lp_kind"], run["status"], float_string(run["solver_objective"]),
                run["outcome_kind"], run["candidate_id"], string(run["is_duplicate"]),
                run["classical_bound"], float_string(run["target_value"]),
                float_string(run["gap"]), run["uniform_slack"],
                float_string(run["normalized_gap"]), string(run["target_violates"]),
                float_string(run["critical_visibility"]),
                float_string(run["critical_noise_fraction"]),
                string(run["saturation_count"]), string(run["face_dimension"]),
                string(run["is_facet"]),
            ], '\t'))
        end
    end
end

function write_candidate_table(path, candidates)
    headers = [
        "candidate_id", "first_sample_id", "best_sample_id", "lp_kinds",
        "discovery_count", "bound", "uniform_slack", "cycle_minimum",
        "clockwise_minimum", "counterclockwise_minimum", "saturation_count",
        "face_dimension", "is_facet", "best_target_value", "best_gap",
        "best_normalized_gap", "best_critical_visibility",
        "best_critical_noise_fraction", "coefficients_in_canonical_64_order",
    ]
    atomic_write(path) do io
        println(io, join(headers, '\t'))
        for candidate in candidates
            println(io, join([
                candidate["candidate_id"], string(candidate["first_sample_id"]),
                string(candidate["best_sample_id"]), join(candidate["lp_kinds"], ","),
                string(candidate["discovery_count"]), string(candidate["bound"]),
                rational_string(candidate["uniform_slack"]),
                string(candidate["cycle_minimum"]), string(candidate["clockwise_minimum"]),
                string(candidate["counterclockwise_minimum"]),
                string(candidate["saturation_count"]), string(candidate["face_dimension"]),
                string(candidate["is_facet"]), float_string(candidate["best_target_value"]),
                float_string(candidate["best_gap"]), float_string(candidate["best_normalized_gap"]),
                float_string(candidate["best_critical_visibility"]),
                float_string(candidate["best_critical_noise_fraction"]),
                integer_vector_string(candidate["coefficients"]),
            ], '\t'))
        end
    end
end

function write_ranking_table(path, candidates)
    violating = filter(candidate -> candidate["best_normalized_gap"] > NORMALIZED_VIOLATION_TOLERANCE, candidates)
    ranking = sort(violating; by=candidate -> candidate["best_critical_noise_fraction"], rev=true)

    atomic_write(path) do io
        println(io, join([
            "rank", "candidate_id", "is_facet", "face_dimension", "lp_kinds",
            "best_sample_id", "bound", "best_target_value", "best_normalized_gap",
            "best_critical_visibility", "best_critical_noise_fraction",
        ], '\t'))
        for (rank_index, candidate) in enumerate(ranking)
            println(io, join([
                string(rank_index), candidate["candidate_id"], string(candidate["is_facet"]),
                string(candidate["face_dimension"]), join(candidate["lp_kinds"], ","),
                string(candidate["best_sample_id"]), string(candidate["bound"]),
                float_string(candidate["best_target_value"]),
                float_string(candidate["best_normalized_gap"]),
                float_string(candidate["best_critical_visibility"]),
                float_string(candidate["best_critical_noise_fraction"]),
            ], '\t'))
        end
    end
end


write_ranking_table (generic function with 1 method)

In [24]:
function write_realization_tables(paths, samples)
    atomic_write(paths["states"]) do io
        println(io, "sample_id\tbasis_index\ta\tb\tc\treal\timag")
        for sample in samples, (a, b, c) in TRIPLES
            index = state_index(a, b, c)
            amplitude = sample["state"][index]
            println(io, join([
                string(sample["sample_id"]), string(index), string(a), string(b), string(c),
                float_string(real(amplitude)), float_string(imag(amplitude)),
            ], '\t'))
        end
    end

    atomic_write(paths["measurements"]) do io
        println(io, "sample_id\tparty\tinput\toutput\trow\tcolumn\treal\timag")
        for sample in samples, party in 1:3, input in 0:1, output in 0:1, row in 1:2, column in 1:2
            entry = sample["measurements"][party][input + 1][output + 1][row, column]
            println(io, join([
                string(sample["sample_id"]), string(party), string(input), string(output),
                string(row), string(column), float_string(real(entry)), float_string(imag(entry)),
            ], '\t'))
        end
    end

    atomic_write(paths["behaviors"]) do io
        println(io, "sample_id\tx\ty\tz\ta\tb\tc\tprobability")
        for sample in samples, (x, y, z) in TRIPLES, (a, b, c) in TRIPLES
            probability = sample["behavior"][prob_index(a, b, c, x, y, z)]
            println(io, join([
                string(sample["sample_id"]), string(x), string(y), string(z),
                string(a), string(b), string(c), float_string(probability),
            ], '\t'))
        end
    end
end

function write_metadata(path, state, vertex_fingerprint)
    config = state["config"]
    atomic_write(path) do io
        println(io, "Blind near-GHZ Acín quantum-target search")
        println(io, "saved_at = ", now())
        println(io, "julia_version = ", VERSION)
        println(io, "format_version = ", state["format_version"])
        println(io, "master_seed = ", config["master_seed"])
        println(io, "near_ghz_acin_samples = ", config["n_near_ghz"])
        println(io, "samples_per_r_stratum = ", config["samples_per_stratum"])
        println(io, "r_strata = ", join(string.(config["r_strata"]), "; "))
        println(io, "state_parameterization = ", config["state_parameterization"])
        println(io, "state_sampling = ", config["state_sampling"])
        println(io, "measurement_scheme = ", config["measurement_scheme"])
        println(io, "completed_samples = ", length(state["samples"]))
        println(io, "lp_runs = ", length(state["lp_runs"]))
        println(io, "unique_candidates = ", length(state["candidates"]))
        println(io, "failures = ", length(state["failures"]))
        println(io, "vertex_count = ", size(vertices_cycle, 1))
        println(io, "vertex_fingerprint_sha256 = ", vertex_fingerprint)
        println(io, "coordinate_order = 1+c+2b+4a+8z+16y+32x")
        println(io, "fixed_beta_lp = min c.q; c.v>=1; c>=0")
        println(io, "free_beta_lp = min c.q-beta; c.v>=beta; c.u-beta=1; c,beta free")
        println(io, "rational_reconstruction_tolerance = ", config["rational_tolerance"])
        println(io, "lp_consistency_tolerance = ", config["lp_consistency_tolerance"])
        println(io, "quantum_audit_tolerance = ", config["quantum_audit_tolerance"])
        println(io, "normalized_violation_tolerance = ", config["normalized_violation_tolerance"])
    end
end

function write_all_tables!(paths, state, vertex_fingerprint)
    write_sample_table(paths["samples"], state["samples"])
    write_lp_run_table(paths["lp_runs"], state["lp_runs"])
    write_candidate_table(paths["candidates"], state["candidates"])
    write_ranking_table(paths["ranking"], state["candidates"])
    write_realization_tables(paths, state["samples"])
    write_metadata(paths["metadata"], state, vertex_fingerprint)
    return nothing
end


write_all_tables! (generic function with 1 method)

## 10. Search engine

Each sample has an independent seed. The sample is processed as a transaction:

1. generate and audit the realization;
2. solve and audit the fixed-$\beta$ LP;
3. solve and audit the free-$\beta$ LP;
4. only then append the sample and both results to the persistent state.

If anything fails before step 4, the default fail-fast run records the error and leaves that sample as the next one to retry after correction. This avoids incomplete samples in the result tables.

In [26]:
function vertex_fingerprint(vertices)
    bytes = reinterpret(UInt8, vec(copy(vertices)))
    return bytes2hex(sha256(bytes))
end

const VERTEX_FINGERPRINT = vertex_fingerprint(vertices_cycle)

function search_config()
    return Dict{String, Any}(
        "master_seed" => MASTER_SEED,
        "n_near_ghz" => N_NEAR_GHZ_SAMPLES,
        "r_strata" => [
            (stratum.label, stratum.lower, stratum.upper)
            for stratum in R_STRATA
        ],
        "samples_per_stratum" => div(N_NEAR_GHZ_SAMPLES, length(R_STRATA)),
        "state_parameterization" =>
            "Acin: l0|000>+l1 exp(i phi)|100>+l2|101>+l3|110>+l4|111>",
        "state_sampling" =>
            "r stratified uniform; s uniform; (t1,t2,t3) Dirichlet(1,1,1); phi uniform [0,pi]",
        "measurement_scheme" =>
            "input 0 computational for every party; input 1 independent Haar projective",
        "rational_tolerance" => RATIONAL_RECONSTRUCTION_TOLERANCE,
        "lp_consistency_tolerance" => LP_CONSISTENCY_TOLERANCE,
        "quantum_audit_tolerance" => QUANTUM_AUDIT_TOLERANCE,
        "normalized_violation_tolerance" => NORMALIZED_VIOLATION_TOLERANCE,
        "vertex_fingerprint" => VERTEX_FINGERPRINT,
        "coordinate_order" => "1+c+2b+4a+8z+16y+32x",
    )
end

function fresh_state()
    return Dict{String, Any}(
        "format_version" => 2,
        "config" => search_config(),
        "next_sample_id" => 1,
        "samples" => Any[],
        "lp_runs" => Any[],
        "candidates" => Any[],
        "candidate_index" => Dict{String, Int}(),
        "failures" => Any[],
    )
end

function validate_checkpoint(state)
    state["format_version"] == 2 || error("Checkpoint format mismatch")
    state["config"] == search_config() || error(
        "Checkpoint configuration mismatch. Use the same configuration or a new output directory.",
    )
    all(
        sample["sample_id"] < state["next_sample_id"]
        for sample in state["samples"]
    ) || error("Checkpoint sample index is inconsistent")
    length(state["lp_runs"]) == 2 * length(state["samples"]) || error(
        "Checkpoint does not contain exactly two LP results per committed sample",
    )
    return true
end

"""Deterministic assignment of one sample ID to one r stratum and seed."""
function sample_spec(sample_id)
    1 <= sample_id <= N_NEAR_GHZ_SAMPLES ||
        error("Sample ID is outside the configured plan")

    number_of_strata = length(R_STRATA)
    stratum_index = 1 + mod(sample_id - 1, number_of_strata)
    stratum_draw_index = 1 + div(sample_id - 1, number_of_strata)
    seed = MASTER_SEED + sample_id - 1
    return (
        ensemble="near_ghz_acin_fixed_input0",
        ensemble_index=sample_id,
        seed=seed,
        stratum_index=stratum_index,
        stratum_draw_index=stratum_draw_index,
        stratum=R_STRATA[stratum_index],
    )
end

function generate_sample(sample_id)
    spec = sample_spec(sample_id)
    rng = MersenneTwister(spec.seed)
    quantum_state, measurements, parameters = near_ghz_acin_realization(
        rng,
        spec.stratum,
    )

    behavior, imaginary_error = quantum_behavior(quantum_state, measurements)
    audit = audit_quantum_realization(
        quantum_state,
        measurements,
        behavior,
        imaginary_error,
    )

    return Dict{String, Any}(
        "sample_id" => sample_id,
        "ensemble" => spec.ensemble,
        "ensemble_index" => spec.ensemble_index,
        "seed" => spec.seed,
        "r_stratum_index" => spec.stratum_index,
        "r_stratum_label" => spec.stratum.label,
        "stratum_draw_index" => spec.stratum_draw_index,
        "r_lower" => spec.stratum.lower,
        "r_upper" => spec.stratum.upper,
        "parameters" => parameters,
        "state" => quantum_state,
        "measurements" => measurements,
        "behavior" => behavior,
        "audit" => audit,
    )
end

function solve_both_lps(sample)
    target = sample["behavior"]

    fixed_numerical = solve_fixed_beta_target!(
        fixed_beta_model,
        fixed_beta_coefficients,
        target,
    )
    fixed_outcome = audit_lp_solution(
        "fixed_beta",
        fixed_numerical,
        target,
        vertices_cycle,
        vertices_clockwise,
        vertices_counterclockwise,
    )

    free_numerical = solve_free_beta_target!(
        free_beta_model,
        free_beta_coefficients,
        free_beta_beta,
        target,
    )
    free_outcome = audit_lp_solution(
        "free_beta",
        free_numerical,
        target,
        vertices_cycle,
        vertices_clockwise,
        vertices_counterclockwise,
    )

    return [fixed_outcome, free_outcome]
end


solve_both_lps (generic function with 1 method)

In [27]:
function target_metrics_for_representative(candidate, target)
    target_value = sum(
        Float64(candidate["coefficients"][j]) * target[j]
        for j in eachindex(target)
    )
    gap = Float64(candidate["bound"]) - target_value
    normalized_gap = gap / Float64(candidate["uniform_slack"])
    target_violates = normalized_gap > NORMALIZED_VIOLATION_TOLERANCE
    critical_visibility = target_violates ? 1 / (1 + normalized_gap) : NaN
    critical_noise_fraction = target_violates ? normalized_gap / (1 + normalized_gap) : NaN
    return (
        target_value=target_value,
        gap=gap,
        normalized_gap=normalized_gap,
        critical_visibility=critical_visibility,
        critical_noise_fraction=critical_noise_fraction,
    )
end

function update_unique_candidate!(state, sample, outcome)
    candidate = outcome["candidate"]
    candidate === nothing && return "", false

    key = candidate["canonical_slack_key"]
    index = get(state["candidate_index"], key, 0)
    is_duplicate = index != 0

    if !is_duplicate
        stored = deepcopy(candidate)
        metrics = target_metrics_for_representative(stored, sample["behavior"])
        stored["first_sample_id"] = sample["sample_id"]
        stored["best_sample_id"] = sample["sample_id"]
        stored["lp_kinds"] = String[outcome["lp_kind"]]
        stored["discovery_count"] = 1
        stored["best_target_value"] = metrics.target_value
        stored["best_gap"] = metrics.gap
        stored["best_normalized_gap"] = metrics.normalized_gap
        stored["best_critical_visibility"] = metrics.critical_visibility
        stored["best_critical_noise_fraction"] = metrics.critical_noise_fraction
        push!(state["candidates"], stored)
        index = length(state["candidates"])
        state["candidate_index"][key] = index
    else
        stored = state["candidates"][index]
        metrics = target_metrics_for_representative(stored, sample["behavior"])
        relative_agreement(metrics.normalized_gap, outcome["normalized_gap"]) || error(
            "Gauge-equivalent representatives disagree on normalized target slack",
        )
        stored["discovery_count"] += 1
        if !(outcome["lp_kind"] in stored["lp_kinds"])
            push!(stored["lp_kinds"], outcome["lp_kind"])
            sort!(stored["lp_kinds"])
        end
        if metrics.normalized_gap > stored["best_normalized_gap"]
            stored["best_sample_id"] = sample["sample_id"]
            stored["best_target_value"] = metrics.target_value
            stored["best_gap"] = metrics.gap
            stored["best_normalized_gap"] = metrics.normalized_gap
            stored["best_critical_visibility"] = metrics.critical_visibility
            stored["best_critical_noise_fraction"] = metrics.critical_noise_fraction
        end
    end

    return state["candidates"][index]["candidate_id"], is_duplicate
end

function make_lp_run_record(sample, outcome, candidate_id, is_duplicate)
    candidate = outcome["candidate"]
    return Dict{String, Any}(
        "sample_id" => sample["sample_id"],
        "ensemble" => sample["ensemble"],
        "seed" => sample["seed"],
        "lp_kind" => outcome["lp_kind"],
        "status" => outcome["status"],
        "solver_objective" => outcome["solver_objective"],
        "outcome_kind" => outcome["outcome_kind"],
        "candidate_id" => candidate_id,
        "is_duplicate" => is_duplicate,
        "classical_bound" => candidate === nothing ? "" : string(candidate["bound"]),
        "target_value" => outcome["target_value"],
        "gap" => outcome["gap"],
        "uniform_slack" => candidate === nothing ? "" : rational_string(candidate["uniform_slack"]),
        "normalized_gap" => outcome["normalized_gap"],
        "target_violates" => outcome["target_violates"],
        "critical_visibility" => outcome["critical_visibility"],
        "critical_noise_fraction" => outcome["critical_noise_fraction"],
        "saturation_count" => candidate === nothing ? 0 : candidate["saturation_count"],
        "face_dimension" => candidate === nothing ? -1 : candidate["face_dimension"],
        "is_facet" => candidate === nothing ? false : candidate["is_facet"],
    )
end

"""Commit one completely processed sample and its two LP outcomes."""
function commit_sample!(state, sample, outcomes)
    @assert length(outcomes) == 2
    @assert Set(outcome["lp_kind"] for outcome in outcomes) == Set(["fixed_beta", "free_beta"])

    push!(state["samples"], sample)
    for outcome in outcomes
        candidate_id, is_duplicate = update_unique_candidate!(state, sample, outcome)
        push!(
            state["lp_runs"],
            make_lp_run_record(sample, outcome, candidate_id, is_duplicate),
        )
    end
    state["next_sample_id"] = sample["sample_id"] + 1
    return nothing
end

commit_sample!

In [28]:
function run_blind_search!(;
    output_directory=SEARCH_OUTPUT_DIRECTORY,
    resume=RESUME_SEARCH,
    stop_on_error=STOP_ON_ERROR,
    write_tables_every=WRITE_TABLES_EVERY,
)
    mkpath(output_directory)
    paths = output_paths(output_directory)

    state = if resume && isfile(paths["checkpoint"])
        loaded = read_checkpoint(paths["checkpoint"])
        validate_checkpoint(loaded)
        println(
            "Resuming at sample ", loaded["next_sample_id"],
            " after ", length(loaded["samples"]), " completed samples.",
        )
        loaded
    else
        if !resume && any(isfile(path) for path in values(paths))
            error(
                "RESUME_SEARCH=false but the output directory contains result files. " *
                "Choose a new output directory to avoid overwriting a research run.",
            )
        end
        fresh_state()
    end

    total_samples = N_NEAR_GHZ_SAMPLES
    started_at = time()

    try
        while state["next_sample_id"] <= total_samples
            sample_id = state["next_sample_id"]
            spec = sample_spec(sample_id)
            println(
                "\nSample $sample_id/$total_samples | ",
                spec.stratum.label,
                " #", spec.stratum_draw_index,
                " | seed=", spec.seed,
            )

            try
                # No persistent state is mutated until generation, both solves,
                # and all exact audits have succeeded.
                sample = generate_sample(sample_id)
                parameters = sample["parameters"]
                @printf(
                    "  state      | r = %.9e | tau = %.9e\n",
                    parameters["r"],
                    parameters["three_tangle"],
                )

                outcomes = solve_both_lps(sample)
                commit_sample!(state, sample, outcomes)

                for outcome in outcomes
                    label = outcome["outcome_kind"] == "affine_equality" ?
                        "affine equality" :
                        (outcome["candidate"]["is_facet"] ? "facet" : "nonfacet")
                    @printf(
                        "  %-10s | %-15s | normalized gap = % .9e | noise = %s\n",
                        outcome["lp_kind"],
                        label,
                        outcome["normalized_gap"],
                        float_string(outcome["critical_noise_fraction"]),
                    )
                end

                write_checkpoint(paths["checkpoint"], state)
                if sample_id % write_tables_every == 0 || sample_id == total_samples
                    write_all_tables!(paths, state, VERTEX_FINGERPRINT)
                end
            catch error_value
                error_value isa InterruptException && rethrow()
                failure = Dict{String, Any}(
                    "timestamp" => string(now()),
                    "sample_id" => sample_id,
                    "ensemble" => spec.ensemble,
                    "stage" => "generate_solve_or_audit",
                    "message" => sprint(showerror, error_value),
                )
                push!(state["failures"], failure)
                append_failure(
                    paths["failures"],
                    sample_id,
                    spec.ensemble,
                    failure["stage"],
                    error_value,
                )
                write_checkpoint(paths["checkpoint"], state)

                if stop_on_error
                    rethrow()
                else
                    @warn "Skipping failed sample" sample_id exception=(error_value, catch_backtrace())
                    state["next_sample_id"] = sample_id + 1
                    write_checkpoint(paths["checkpoint"], state)
                end
            end
        end
    finally
        write_checkpoint(paths["checkpoint"], state)
        write_all_tables!(paths, state, VERTEX_FINGERPRINT)
    end

    elapsed_minutes = (time() - started_at) / 60
    violating_runs = count(run -> run["target_violates"], state["lp_runs"])
    violating_candidates = count(
        candidate -> candidate["best_normalized_gap"] > NORMALIZED_VIOLATION_TOLERANCE,
        state["candidates"],
    )
    facet_candidates = count(candidate -> candidate["is_facet"], state["candidates"])

    println("\nBlind batch complete")
    println("  completed quantum samples: ", length(state["samples"]))
    println("  LP runs:                   ", length(state["lp_runs"]))
    println("  target-violating LP runs:  ", violating_runs)
    println("  distinct inequalities:     ", length(state["candidates"]))
    println("  distinct facets:           ", facet_candidates)
    println("  violating inequalities:    ", violating_candidates)
    @printf("  elapsed this call:          %.2f minutes\n", elapsed_minutes)
    println("  results directory:          ", abspath(output_directory))

    return state
end


run_blind_search! (generic function with 1 method)

## 11. Run or resume the blind batch

This is the only long-running cell. Interrupting it with Jupyter's stop button still triggers the `finally` block, which writes the current checkpoint and tables. Re-running the cell resumes from the next uncommitted sample.

In [30]:
search_state = run_blind_search!()


Sample 1/40000 | r_0_to_0.01 #1 | seed=42000001
  state      | r = 5.048207719e-03 | tau = 5.434818699e-01
  fixed_beta | affine equality | normalized gap =  NaN | noise = 
  free_beta  | nonfacet        | normalized gap =  0.000000000e+00 | noise = 

Sample 2/40000 | r_0.01_to_0.05 #1 | seed=42000002
  state      | r = 1.497566804e-02 | tau = 8.665624658e-01
  fixed_beta | affine equality | normalized gap =  NaN | noise = 
  free_beta  | nonfacet        | normalized gap =  0.000000000e+00 | noise = 

Sample 3/40000 | r_0.05_to_0.15 #1 | seed=42000003
  state      | r = 5.235653164e-02 | tau = 7.191501987e-01
  fixed_beta | affine equality | normalized gap =  NaN | noise = 
  free_beta  | nonfacet        | normalized gap = -8.881784197e-16 | noise = 

Sample 4/40000 | r_0.15_to_0.40 #1 | seed=42000004
  state      | r = 3.519652849e-01 | tau = 4.026651029e-01
  fixed_beta | affine equality | normalized gap =  NaN | noise = 
  free_beta  | nonfacet        | normalized gap =  0.00000000

Excessive output truncated after 524341 bytes.


Sample 2025/40000 | r_0_to_0.01 #507 | seed=42002025


Dict{String, Any} with 8 entries:
  "format_version"  => 2
  "candidate_index" => Dict("1,4,1,4,1,4,1,4,0,3,0,3,0,3,0,3,4,6,4,6,4,6,4,6,3,…
  "config"          => Dict{String, Any}("samples_per_stratum"=>10000, "state_p…
  "next_sample_id"  => 40001
  "samples"         => Any[Dict{String, Any}("r_stratum_index"=>1, "r_stratum_l…
  "lp_runs"         => Any[Dict{String, Any}("outcome_kind"=>"affine_equality",…
  "failures"        => Any[]
  "candidates"      => Any[Dict{String, Any}("best_critical_visibility"=>NaN, "…

## 12. Inspect the ranking in Julia

The TSV files are the primary portable outputs. This final cell also prints the best distinct candidates found in the current run. A positive normalized gap means the sampled quantum behaviour violates the reconstructed exact inequality. NPA and dedicated see-saw optimization are intentionally deferred until this blind batch is complete.

In [32]:
violating_candidates = filter(
    candidate -> candidate["best_normalized_gap"] > NORMALIZED_VIOLATION_TOLERANCE,
    search_state["candidates"],
)
ranked_candidates = sort(
    violating_candidates;
    by=candidate -> candidate["best_critical_noise_fraction"],
    rev=true,
)

println("\nBest distinct inequalities from this blind batch")
println("rank  facet  face-dim  LP source(s)           sample   normalized gap   white-noise fraction")

for (rank_index, candidate) in enumerate(first(ranked_candidates, min(20, length(ranked_candidates))))
    @printf(
        "%4d  %5s  %8d  %-21s  %6d   % .9e   % .9e\n",
        rank_index,
        string(candidate["is_facet"]),
        candidate["face_dimension"],
        join(candidate["lp_kinds"], ","),
        candidate["best_sample_id"],
        candidate["best_normalized_gap"],
        candidate["best_critical_noise_fraction"],
    )
end

isempty(ranked_candidates) && println("  No target violation was found in the configured batch.")
println("\nPortable ranking file: ", abspath(output_paths(SEARCH_OUTPUT_DIRECTORY)["ranking"]))


Best distinct inequalities from this blind batch
rank  facet  face-dim  LP source(s)           sample   normalized gap   white-noise fraction
   1   true        49  fixed_beta,free_beta    19181    5.742059703e-03    5.709276696e-03
   2  false        48  fixed_beta              19181    5.740043113e-03    5.707283062e-03
   3   true        49  fixed_beta,free_beta     8729    5.458824283e-03    5.429187304e-03
   4  false        48  fixed_beta               8729    5.448181802e-03    5.418659958e-03
   5   true        49  fixed_beta,free_beta    24385    5.149784992e-03    5.123400580e-03
   6  false        48  fixed_beta              24385    5.141255753e-03    5.114958443e-03
   7   true        49  fixed_beta,free_beta    14789    5.119587338e-03    5.093510665e-03
   8  false        48  fixed_beta              14789    5.119555746e-03    5.093479394e-03
   9   true        49  fixed_beta,free_beta     4441    5.040752608e-03    5.015470860e-03
  10  false        48  fixed_beta     

## Interpretation and next stage

A successful row in `candidate_ranking.tsv` establishes the following, within the recorded numerical quantum evaluation:

- the coefficient vector and bound define an inequality valid on **all** 8,128 corrected cycle vertices;
- the exact face dimension states whether the inequality is a facet;
- the saved Acín parameters, state amplitudes, and projectors regenerate the target behaviour by the Born rule;
- `best_critical_noise_fraction` compares candidates in a scale-independent way for their best sampled target.

The search distribution is deliberately biased: the state is near the generalized-GHZ subspace and input 0 is fixed in the canonical computational basis. Therefore, a failure to find a new witness is evidence about this ensemble, not a no-go theorem for arbitrary states and measurements.

The appropriate next notebook should take only the most promising distinct candidates, recompute their quantum values at high precision, run NPA upper bounds, and perform dedicated multistart see-saws. Keeping that optimization outside this notebook preserves the blind-discovery logic: the random batch is not steered by recognizing an inequality found partway through it.
